In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image, ImageChops

from sklearn.metrics import cohen_kappa_score
import timm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # enable cudnn auto‑tuning for faster conv ops




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
threshold = [0.7, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = torch.zeros(out.size(0))
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu()
    return prediction




In [3]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return f"{self.__class__.__name__}(p={self.p.data.item():.4f}, eps={self.eps})"


class ThreeStage_Model(nn.Module):
    def __init__(self):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.create_model("tf_efficientnet_b4_ns", pretrained=True)
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [4]:
class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w
        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h
        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        else:
            return image




In [5]:
train_csv_path = "../input/aptos2019-blindness-detection/train.csv"
test_csv_path = "../input/aptos2019-blindness-detection/test.csv"
train_img_dir = "../input/aptos2019-blindness-detection/train_images"
test_img_dir = "../input/aptos2019-blindness-detection/test_images"

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

train_ids = train_df["id_code"].values
train_labels = train_df["diagnosis"].values
test_ids = test_df["id_code"].values

input_size = 300
transform = transforms.Compose(
    [
        trim(),
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

net = ThreeStage_Model()
net = net.to(device)




/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [6]:
for param in net.backbone.parameters():
    param.requires_grad = False


class AptosDataset(Dataset):
    def __init__(self, ids, img_dir, transform):
        self.ids = ids
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_code = self.ids[idx]
        img_path = f"{self.img_dir}/{id_code}.png"
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        return img, id_code


raw_dataset = AptosDataset(train_ids, train_img_dir, transform)
raw_loader = DataLoader(
    raw_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True
)

print("Pre‑computing backbone features for training set...")
feat_list = []
with torch.no_grad():
    for imgs, _ in raw_loader:
        imgs = imgs.to(device)
        feats = net.backbone(imgs)  # (B, 1000)
        feat_list.append(feats.cpu())
feat_tensor = torch.cat(feat_list, dim=0)
label_tensor = torch.tensor(train_labels, dtype=torch.long)

train_feat_dataset = TensorDataset(feat_tensor, label_tensor)
train_feat_loader = DataLoader(
    train_feat_dataset, batch_size=32, shuffle=True, num_workers=0
)

class_counts = np.bincount(train_labels, minlength=5)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights * (class_weights.mean())
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.Adam(net.classifier.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)

val_frac = 0.10
val_size = int(len(train_feat_dataset) * val_frac)
train_size = len(train_feat_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(
    train_feat_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(seed),
)

val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0)

net.train()
best_qwk = -np.inf
best_state_dict = None
use_argmax = False  # will be set according to best epoch

num_epochs = 180  # extended training
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for feats, labels in train_feat_loader:
        feats = feats.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = net.classifier(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_feat_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

    net.eval()
    all_labels = []
    exp_preds = []
    argmax_preds = []
    with torch.no_grad():
        for feats, labels in val_loader:
            feats = feats.to(device)
            logits = net.classifier(feats)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            expected = np.dot(probs, np.arange(5))
            pred_exp = np.rint(expected).astype(int)
            pred_exp = np.clip(pred_exp, 0, 4)
            pred_arg = np.argmax(probs, axis=1)

            all_labels.extend(labels.numpy())
            exp_preds.extend(pred_exp)
            argmax_preds.extend(pred_arg)

    qwk_exp = cohen_kappa_score(all_labels, exp_preds, weights="quadratic")
    qwk_arg = cohen_kappa_score(all_labels, argmax_preds, weights="quadratic")
    epoch_best_qwk = max(qwk_exp, qwk_arg)
    if epoch_best_qwk > best_qwk:
        best_qwk = epoch_best_qwk
        best_state_dict = net.classifier.state_dict()
        use_argmax = qwk_arg > qwk_exp
        print(f"  New best QWK on validation: {best_qwk:.5f} (epoch {epoch+1})")
    net.train()
print("Training completed.")
print(f"Best validation QWK achieved: {best_qwk:.5f}")

if best_state_dict is not None:
    net.classifier.load_state_dict(best_state_dict)
else:
    print("Warning: No best state captured; using final model.")




Pre‑computing backbone features for training set...
Epoch 1/180 - Loss: 1.3902
  New best QWK on validation: 0.61598 (epoch 1)
Epoch 2/180 - Loss: 1.1275
  New best QWK on validation: 0.71635 (epoch 2)
Epoch 3/180 - Loss: 0.9993
  New best QWK on validation: 0.72341 (epoch 3)
Epoch 4/180 - Loss: 0.9080
  New best QWK on validation: 0.79223 (epoch 4)
Epoch 5/180 - Loss: 0.8165
  New best QWK on validation: 0.85354 (epoch 5)
Epoch 6/180 - Loss: 0.7356
  New best QWK on validation: 0.85366 (epoch 6)
Epoch 7/180 - Loss: 0.6912
  New best QWK on validation: 0.87694 (epoch 7)
Epoch 8/180 - Loss: 0.6403
  New best QWK on validation: 0.89767 (epoch 8)
Epoch 9/180 - Loss: 0.6049
Epoch 10/180 - Loss: 0.5677
  New best QWK on validation: 0.91870 (epoch 10)
Epoch 11/180 - Loss: 0.5136
  New best QWK on validation: 0.93372 (epoch 11)
Epoch 12/180 - Loss: 0.4908
  New best QWK on validation: 0.94422 (epoch 12)
Epoch 13/180 - Loss: 0.4661
Epoch 14/180 - Loss: 0.4443
  New best QWK on validation: 0.95

In [7]:
print(
    "Running inference on test images with horizontal‑flip and vertical‑flip TTA (classifier only)..."
)
submission = []
batch_size = 64


def predict_batch(ids):
    # load and transform original images
    imgs = []
    for idx in ids:
        img_path = f"{test_img_dir}/{idx}.png"
        img = Image.open(img_path).convert("RGB")
        img = transform(img)
        imgs.append(img)
    imgs_tensor = torch.stack(imgs).to(device)

    # create flipped versions for TTA
    imgs_h = torch.stack([TF.hflip(img) for img in imgs]).to(device)
    imgs_v = torch.stack([TF.vflip(img) for img in imgs]).to(device)

    with torch.no_grad():
        # original
        feats = net.backbone(imgs_tensor)
        logits = net.classifier(feats)
        probs = torch.softmax(logits, dim=1)

        # horizontal flip
        feats_h = net.backbone(imgs_h)
        logits_h = net.classifier(feats_h)
        probs_h = torch.softmax(logits_h, dim=1)

        # vertical flip
        feats_v = net.backbone(imgs_v)
        logits_v = net.classifier(feats_v)
        probs_v = torch.softmax(logits_v, dim=1)

        # average probabilities across augmentations
        avg_probs = (probs + probs_h + probs_v) / 3.0
        avg_probs_np = avg_probs.cpu().numpy()

        if use_argmax:
            cls_pred = np.argmax(avg_probs_np, axis=1)
        else:
            expected = np.dot(avg_probs_np, np.arange(5))
            cls_pred = np.rint(expected).astype(int)

        cls_pred = np.clip(cls_pred, 0, 4)

    return cls_pred


batch = []
for i, idx in enumerate(test_ids):
    batch.append(idx)
    if len(batch) == batch_size or i == len(test_ids) - 1:
        preds = predict_batch(batch)
        for pid, pred in zip(batch, preds):
            submission.append([pid, int(pred)])
        batch = []

submission = np.array(submission)




Running inference on test images with horizontal‑flip and vertical‑flip TTA (classifier only)...


In [8]:
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
df.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' written.")

Submission file 'submission.csv' written.
